# 📖 Sentiment Analysis using LSTM & Hidden State Visualizer

This notebook trains a **Long Short-Term Memory (LSTM)** network on the full `sentiment_analysis.csv` dataset.
It includes:
1. Full dataset loading & preprocessing
2. Multi-class sentiment classification (`negative`, `neutral`, `positive`)
3. Keras Tokenization & Sequence Padding
4. LSTM Architecture Training with Dropout Regularization
5. Timestep Hidden State Inspection & Visualization (`return_sequences=True` Weight Sharing)
6. Artifact Export (`sentiment_lstm.weights.h5`, `tokenizer.pkl`, `config.pkl`) for FastAPI & Streamlit/Vercel deployment.

In [ ]:
import numpy as np
import pandas as pd
import pickle
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout

print("TensorFlow Version:", tf.__version__)

## 1. Load Dataset

In [ ]:
# Load dataset from local file or repository raw URL
try:
    df = pd.read_csv("sentiment_analysis.csv")
except Exception:
    url = "https://raw.githubusercontent.com/Tanishk-Rastogi/Sentiment-Analysis-using-RNN/main/sentiment_analysis.csv"
    df = pd.read_csv(url)

print("Dataset Shape:", df.shape)
df.head()

## 2. Preprocess & Encode Labels

In [ ]:
# Drop rows with missing text or sentiment
df = df.dropna(subset=['text', 'sentiment']).copy()

# Standardize text and sentiment strings
df['text'] = df['text'].astype(str).str.strip()
df['sentiment'] = df['sentiment'].astype(str).str.strip().str.lower()

# Map sentiment labels to integers
label_mapping = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['sentiment'].map(label_mapping)

# Drop any unmapped sentiments if present
df = df.dropna(subset=['label']).copy()
df['label'] = df['label'].astype(int)

print("Sentiment Distribution:")
print(df['sentiment'].value_counts())
print("\nTotal Samples:", len(df))

## 3. Tokenization & Sequence Padding

In [ ]:
vocab_size = 5000
oov_tok = '<OOV>'

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok, lower=True)
tokenizer.fit_on_texts(df['text'])

sequences = tokenizer.texts_to_sequences(df['text'])
maxlen = 35
print(f"Vocabulary Size: {len(tokenizer.word_index) + 1}")
print(f"Max Sequence Length: {maxlen}")

X = pad_sequences(sequences, maxlen=maxlen, padding='post', truncating='post')
y = df['label'].values

print("X shape:", X.shape)
print("y shape:", y.shape)

## 4. Build LSTM Model

In [ ]:
embed_dim = 64
lstm_units = 32
num_classes = 3

inp = Input(shape=(maxlen,), dtype="int32", name='input')
x = Embedding(input_dim=vocab_size, output_dim=embed_dim, mask_zero=True, name='embed')(inp)
x = Dropout(0.2)(x)
lstm = LSTM(units=lstm_units, return_sequences=False, return_state=False, name='lstm_layer')(x)
x_out = Dropout(0.2)(lstm)
out = Dense(num_classes, activation='softmax', name='out')(x_out)

model = Model(inputs=inp, outputs=out)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.003), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 5. Train Model

In [ ]:
epochs = 25
batch_size = 16

history = model.fit(
    X, y,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.15,
    verbose=1
)

## 6. Inspect Hidden States (Timestep by Timestep)

In [ ]:
from tensorflow.keras.layers import LSTM as KerasLSTM

seq_inp = Input(shape=(maxlen,), dtype='int32')
seq_emb = model.get_layer('embed')(seq_inp)

# LSTM configured with return_sequences=True to return hidden states at every timestep
lstm_seq = KerasLSTM(units=lstm_units, return_sequences=True, name='lstm_seq')
seq_hidden = lstm_seq(seq_emb)

# Copy weights from trained LSTM layer
trained_weights = model.get_layer('lstm_layer').get_weights()
lstm_seq.set_weights(trained_weights)

inspect_model = Model(inputs=seq_inp, outputs=seq_hidden)

# Predict hidden states on a sample text
sample_text = "This project is really amazing and super helpful!"
sample_seq = pad_sequences(tokenizer.texts_to_sequences([sample_text]), maxlen=maxlen, padding='post')
hidden_states = inspect_model.predict(sample_seq)

print("Sample Text:", sample_text)
print("Hidden States Shape (batch, maxlen, lstm_units):", hidden_states.shape)
print("Hidden states for first 5 timesteps:")
print(np.round(hidden_states[0][:5], 3))

## 7. Export Model Artifacts for Deployment (FastAPI, Streamlit & Vercel)

In [ ]:
# Save LSTM model weights
model.save_weights("sentiment_lstm.weights.h5")
print("Saved sentiment_lstm.weights.h5")

# Save Tokenizer
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
print("Saved tokenizer.pkl")

# Save Label Mapping & Metadata
config = {
    'label_mapping': label_mapping,
    'inv_label_mapping': {v: k for k, v in label_mapping.items()},
    'maxlen': maxlen,
    'vocab_size': vocab_size,
    'lstm_units': lstm_units,
    'embed_dim': embed_dim
}

with open("config.pkl", "wb") as f:
    pickle.dump(config, f)
print("Saved config.pkl")

# Trigger automatic download if running in Google Colab
try:
    from google.colab import files
    files.download("sentiment_lstm.weights.h5")
    files.download("tokenizer.pkl")
    files.download("config.pkl")
    print("Triggered Colab file downloads.")
except ImportError:
    print("Artifacts saved locally in current directory.")